In [1]:
# Blueprint of Machine Learning - single input, single output

# Generally, all examples in this notebook are based on this flow:
# 1. Calculate prediction based on input and weight
# 2. Measure how much prediction missed the target (error)
# 3. Figure out amount to modify the weight so the error is smaller next time (Gradient descent)
# 4. Repeat until error is below our desired threshold (or we reached max number of iterations)

input = 2
weight = 0.5
target = 0.8
alpha = 0.1 # We use alpha to control the "speed" of learning because, sometimes weight_delta is can get big and overshoot the target which in result can put the whole system into negative feedback loop completely messing up the learning process.

for iteration in range(100):
    prediction = input * weight
    error = (prediction - target) ** 2
    delta = prediction - target
    weight_delta = delta * input
    weight = weight - (alpha * weight_delta)
    print(f"    Predicted value: {prediction:.5f}")
    print(f"    Actual value: {target:.5f}")
    print(f"    Error: {error:.10f}")
    print(f"    Direction and amount: {weight_delta:.5f}\n----")
    if error < 0.01:
        print(f"==== Training finished after {iteration} iterations: error is below 0.1%!")
        break

    Predicted value: 1.00000
    Actual value: 0.80000
    Error: 0.0400000000
    Direction and amount: 0.40000
----
    Predicted value: 0.92000
    Actual value: 0.80000
    Error: 0.0144000000
    Direction and amount: 0.24000
----
    Predicted value: 0.87200
    Actual value: 0.80000
    Error: 0.0051840000
    Direction and amount: 0.14400
----
==== Training finished after 2 iterations: error is below 0.1%!


In [ ]:
# Full Gradient Descent vs Stochastic Gradient Descent vs Batch Gradient Descent

# NOTICE: All examples below are using Stoch Gradient Descent

# Full Gradient Descent - Uses entire training dataset to compute the weight gradient on each iteration
    # 1.Run a forward pass on all training examples
    # 2.Compute the average gradient
    # 3.Update the weights once per iteration (epoch)
# - Stable, smooth convergence
# - Computationally expensive for large datasets
# - One weight update per epoch

# Stochastic Gradient Descent - Computes weight gradient and immediately updates the weights for each training example
#     1. Pick a single example
#     2. Compute its gradient
#     3. Immediately update the weights
# - Much faster per update
# - Noisy updates (zig-zag path)
# - Can escape shallow local minima
# - Many weight updates per epoch

# Batch Gradient Descent - is the middle ground between Full and Stoch Gradient Descent
# it splits the dataset into small batches and update weights on each batch
#     1. Compute gradients for all samples in the batch
#     2. Average (or sum) them
#     3. Update weights once per batch

# | Method          | Samples per update        | Weight updates | Stability   | Speed             |
# |-----------------|---------------------------|----------------|-------------|-------------------|
# | Full GD         | Entire dataset            | 1 per epoch    | Very stable | Slow              |
# | Stoch GD        | 1 sample                  | Many           | Noisy       | Fast              |
# | Batch GD        | Small group (e.g. 10)     | Moderate       | Balanced    | Fast & efficient  |


In [2]:
# Multiple inputs and one output

# In this example we learn how to predict how much user likes the movie based on what genre the movie is

import numpy as np

# Math Utils
def weighted_sum(vector_a, vector_b):
    assert(len(vector_a) == len(vector_b))
    output = 0
    for i in range (len(vector_a)):
        output += vector_a[i] * vector_b[i]
    return output

def elementwise_multiplication(number, vector):
    output = [0,0,0,0]
    assert(len(output) == len(vector))
    for i in range (len(vector)):
        output[i] = number * vector[i]
    return output

# Movie properties: How much comedy, horror, sci-fi, romance it has.
#              [Comedy, Horror, Sci-Fi, Romance]
scary_movie =  [0.8,    0.2,    0.1,    0.3]  # High comedy, low horror, little romance

# User's rating after watching the movie
actual_user_rating = 8.5 # he rated it 8.5 out of 10.

# User Preferences: We will build user genre preferences based on his rating for what type of movie he watched
#              [Comedy, Horror, Sci-Fi, Romance]
user_profile = [0.0,    0.0,    0.0,    0.0] # Initial weights are zeroed

inputs = scary_movie
weights = user_profile
target = actual_user_rating

print(f"Initial Weights: {weights}")

print(f"\n=== Begin training the model:")

for iteration in range(100):
    prediction = weighted_sum(inputs, weights)
    error = (prediction - target) ** 2
    delta = prediction - target
    weight_deltas = elementwise_multiplication(delta, inputs)
    alpha = 0.5
    print(f"---\nTarget: {target:.5f} Prediction: {prediction:.5f}, Delta: {delta:.5f} Error: {error:.5f}")
    print(f"Wight Deltas: {weight_deltas}")
    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]
    print(f"Updated Weights: {weights}")
    if error < 0.01:
        print(f"\n=== Training finished after {iteration} error is below 0.1% !")
        break

print(f"\n=== Time to evaluate our model, let see how it will rate new movie: ")
# Test Movie properties:
#                [Comedy, Horror, Sci-Fi, Romance]
scary_movie_II = [0.9,    0.1,    0.2,    0.2]  # High comedy, low horror, little romance
input = scary_movie_II
print(f"Scary Movie II properties: [Comedy, Horror, Sci-Fi, Romance] = {scary_movie_II}")

prediction = weighted_sum(input, weights)
print(f"---\nOur model is predicting that user will rate the move: {prediction:.1f} out of 10")

rating_for_scary_movie_II = 9 # User ranked the sequel 9.0 out of 10. He definitely loves the lore.
print(f"---\nLets pull up actual user's rating for this movie: {rating_for_scary_movie_II}")

print("---\nNow lets calculate error")
target = rating_for_scary_movie_II
error = (prediction - target) ** 2
print(f"\nError: {error:.5f}")
if error < 0.01:
    print("Great success! Our model is pretty damn accurate!")

Initial Weights: [0.0, 0.0, 0.0, 0.0]

=== Begin training the model:
---
Target: 8.50000 Prediction: 0.00000, Delta: -8.50000 Error: 72.25000
Wight Deltas: [-6.800000000000001, -1.7000000000000002, -0.8500000000000001, -2.55]
Updated Weights: [3.4000000000000004, 0.8500000000000001, 0.42500000000000004, 1.275]
---
Target: 8.50000 Prediction: 3.31500, Delta: -5.18500 Error: 26.88422
Wight Deltas: [-4.148, -1.037, -0.5185, -1.5554999999999999]
Updated Weights: [5.474, 1.3685, 0.68425, 2.0527499999999996]
---
Target: 8.50000 Prediction: 5.33715, Delta: -3.16285 Error: 10.00362
Wight Deltas: [-2.53028, -0.63257, -0.316285, -0.9488549999999999]
Updated Weights: [6.73914, 1.684785, 0.8423925, 2.5271774999999996]
---
Target: 8.50000 Prediction: 6.57066, Delta: -1.92934 Error: 3.72235
Wight Deltas: [-1.5434708000000001, -0.38586770000000004, -0.19293385000000002, -0.57880155]
Updated Weights: [7.5108754, 1.87771885, 0.938859425, 2.8165782749999995]
---
Target: 8.50000 Prediction: 7.32310, Delt

In [3]:
# Multiple Input and Multiple Outputs

# This example extends previous one but this time we will predict more than one thing at once.
# This model will tell us how user will rate similar movie, will he complete watching it and if he will come back to re-watch it in the future.

# Math Utils
def weighted_sum(vector_a,vector_b):
    assert(len(vector_a) == len(vector_b))
    output = 0
    for i in range (len(vector_a)):
        output += vector_a[i] * vector_b[i]
    return output

def vector_matrix_multiplication(vector, matrix):
    assert(len(vector) == len(matrix[0]))
    output = [0,0,0,0]
    for i in range(len(matrix)):
        output[i] = weighted_sum(vector, matrix[i])
    return output

# Movie Properties: How much comedy, horror, sci-fi, romance it has.
#             [Comedy, Horror, Sci-Fi, Romance]
scary_movie = [0.8,    0.2,    0.1,    0.3]  # High comedy, low horror, little romance

# Actual User Metrics: What did we actually observe from the user interacting with this movie (scary_movie)

users_metrics = [8.5,   # Rating: User rated 8.5 out of 10
                 9.5,   # Completion: User completed watching 95% of the movie (skipped ending credits)
                 3]     # Replays: User re-watched this movie 3 times

# User Profile Model: What we will expect from a user based on a type of movie we will show him.

#              [Comedy,  Horror,  Sci-Fi, Romance]
user_profile = [[0.0,    0.0,    0.0,     0.0], # [Rating] Weights defining how user rates the movie
                [0.0,    0.0,    0.0,     0.0], # [Completion] Weights defining how much of a movie a user will complete
                [0.0,    0.0,    0.0,     0.0]] # [Replays] Weights defining how many times a user will re-watch

inputs = scary_movie
weights = user_profile
targets = users_metrics

print(f"Before training Weights: {user_profile}")

# Let the training algo run up to max 20 iterations
for iteration in range(20):
    print(f"--- Iteration: {iteration}")
    errors = [0.0,
              0.0,
              0.0]

    deltas = [0.0,
              0.0,
              0.0]
    weight_deltas = [[0.0, 0.0, 0.0, 0.0], # weight_deltas have the shape of user_profile matrix
                     [0.0, 0.0, 0.0, 0.0],
                     [0.0, 0.0, 0.0, 0.0]]
    alpha = 0.5

    # Predict
    predictions = vector_matrix_multiplication(inputs, weights)
    print(predictions)

    # Calculate error delta for each target and update weights for each target
    for i in range(len(targets)):
        errors[i] = (predictions[i] - targets[i]) ** 2
        deltas[i] = predictions[i] - targets[i]
        for j in range(len(inputs)):
            weight_deltas[i][j] = deltas[i] * inputs[j]
            weights[i][j] -= alpha * weight_deltas[i][j]

    print(f"Rating Target: {targets[0]:.2f} Predicted: {predictions[0]:.2f}, Delta: {deltas[0]:.3f} Error: {errors[0]:.5f}")
    print(f"Completion Target: {targets[1]:.2f} Predicted: {predictions[1]:.2f}, Delta: {deltas[1]:.3f} Error: {errors[1]:.5f}")
    print(f"Replays Target: {targets[2]:.1f} Predicted: {predictions[2]:.1f}, Delta: {deltas[2]:.3f} Error: {errors[2]:.5f}")

    # Evaluate Error
    if (errors[0] < 0.001) & (errors[1] < 0.001) & (errors[2] < 0.001):
        print(f"=== Training finished after {iteration} iterations All errors are below 0.1% !")
        break

print(f"After training Weights: {weights}")



Before training Weights: [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]
--- Iteration: 0
[0.0, 0.0, 0.0, 0]
Rating Target: 8.50 Predicted: 0.00, Delta: -8.500 Error: 72.25000
Completion Target: 9.50 Predicted: 0.00, Delta: -9.500 Error: 90.25000
Replays Target: 3.0 Predicted: 0.0, Delta: -3.000 Error: 9.00000
--- Iteration: 1
[3.3150000000000004, 3.705, 1.1700000000000002, 0]
Rating Target: 8.50 Predicted: 3.32, Delta: -5.185 Error: 26.88422
Completion Target: 9.50 Predicted: 3.71, Delta: -5.795 Error: 33.58203
Replays Target: 3.0 Predicted: 1.2, Delta: -1.830 Error: 3.34890
--- Iteration: 2
[5.33715, 5.9650500000000015, 1.8837000000000004, 0]
Rating Target: 8.50 Predicted: 5.34, Delta: -3.163 Error: 10.00362
Completion Target: 9.50 Predicted: 5.97, Delta: -3.535 Error: 12.49587
Replays Target: 3.0 Predicted: 1.9, Delta: -1.116 Error: 1.24613
--- Iteration: 3
[6.5706615, 7.3436805, 2.319057, 0]
Rating Target: 8.50 Predicted: 6.57, Delta: -1.929 Error: 3.72235
Comple

In [4]:
%load_ext autoreload
%autoreload 2

# Multiple Input and Multiple Outputs - Real World example: Classifier of handwritten digits
# Version 1.0, Simplified - train on single example and classify only one digit: Digit `5`

# Input size: 784 (grayscale image representing handwritten digit with resolution of 28x28 pixels = 784 inputs)
# Output size: 10 (digits from 0 to 9)
# Training set size: 1 - we will train on only signe image representing handwritten number `5`

import numpy as np # we will use numpy arrays because they have
np.set_printoptions(suppress=True) # supress scientific notation in printing
from sklearn.preprocessing import MinMaxScaler
import resources.mnist.mnist_utils as mnist

# Math Utils
def weighted_sum(vector_a, vector_b):
    assert(len(vector_a) == len(vector_b))
    output = 0
    for i in range (len(vector_a)):
        output += vector_a[i].item() * vector_b[i].item()
    return output

def vector_matrix_multiplication(vector, matrix):
    assert(len(vector) == len(matrix[0]))
    output = np.zeros(len(matrix))
    for i in range(len(matrix)):
        output[i] = weighted_sum(vector, matrix[i])
    return output

print("\n--- Load MNIST dataset")
training_images = mnist.load_training_images()
training_labels = mnist.load_training_labels()

print("\n--- Print one of the images to confirm it's loaded correctly")
image_label = training_labels[0]
print(f"Label for this digit: {image_label}")
image_bitmap = training_images[0]
print(f"image_bitmap shape: {image_bitmap.shape}")
mnist.print_bitmap(image_bitmap)

# Inputs: 2D image bitmap flattened to 1d array (28pixels * 28pixels = 784 inputs)
#              (1st pixel, 2nd pixel,   ...,   last pixel of last row)
# inputs =     [0.8,       0.2,         ...,   0.3]
inputs = MinMaxScaler().fit_transform(image_bitmap.reshape(-1,1)) # Normalize bitmap's grayscale 0-255 values to 0-1.0 range (float)
# It should look something like this
#
#                 ░░░░░░░░▒▒▒▒▓▓░░▓▓████▒▒
#         ░░░░░░▓▓▓▓████████████▓▓████▓▓░░
#       ░░████████████████████░░░░░░░░░░
#       ░░████████████▓▓▓▓████
#         ░░▓▓▒▒██████░░  ░░▓▓
#           ░░░░▓▓██░░
#               ▒▒██▓▓░░
#               ░░▓▓██░░
#                 ░░████▓▓▒▒░░
#                   ░░██████▒▒░░
#                     ░░▓▓████▓▓░░
#                       ░░░░████▓▓
#                           ██████░░
#                     ░░▒▒▓▓██████░░
#                 ░░▒▒██████████▓▓
#             ░░▒▒████████████░░
#         ░░░░██████████▓▓░░░░
#     ░░▓▓██████████▓▓░░░░
# ░░▓▓████████████▒▒░░
# ▒▒████████▒▒▒▒░░

# Weights: Shape of weights = possible answers x matrix is image pixels (10 x 784)
#             1.       2.      3.           784.
# weights =   [[0.4,   0.9,    0.8,   ...,  0.1],    # 1.
#             [0.3,    0.7,    0.9,   ...,  0.3],    # 2.
#             [...,    ...,    ...,   ...,  ...],    ...
#             [0.6,    0.1,    0.3,   ...,  0.5]]    # 10
weights = np.zeros((10,784)) # weights initialized with zeros

# Outputs: 10 outputs, each for every digit from 0 to 9.
# outputs = [0.0, <-- output for digit `0` is 0.0 (0% probability)
#            0.0,
#            0.0, ...
#            0.0,
#            0.0,
#            1.0, <-- output for digit `5` is 1.0 (100% probability)
#            0.0,
#            0.0,
#            0.0,
#            0.0,
#            0.0]
outputs = np.zeros(10)
outputs[image_label] = 1 # set value 1 at the index of image_label = 5

print(f"\n=== Begin training the model:")

for iteration in range(100):
    print(f"--- Iteration: {iteration}")
    errors = np.ones(10)
    deltas = np.zeros(10)
    weight_deltas = np.zeros((10,784))  # weight_deltas matrix have the shape of weights matrix
    alpha = 0.01

    # Predict
    predictions = vector_matrix_multiplication(inputs, weights)

    # Figure out the Error and update Weights for each Target
    for i in range(len(outputs)):
        errors[i] = (predictions[i] - outputs[i]) ** 2
        deltas[i] = predictions[i] - outputs[i]
        for j in range(len(inputs)):
            weight_deltas[i][j] = deltas[i].item() * inputs[j].item()
            weights[i][j] -= alpha * weight_deltas[i][j]

    print(f"Rating Target: {outputs[5]:.2f} Predicted: {predictions[5]:.2f}, Delta: {deltas[5]:.3f} Error: {errors[5]:.5f}")
    print(f"Errors: {errors}")

    # Evaluate Error
    if np.all(errors < 0.01):
        print(f"\n=== Training finished after {iteration} iterations All errors are below 0.1% !")
        break


print("Weights for detecting digit 5:")
mnist.print_weights_image(weights[5].reshape(28,28))

print("\n==== Test time! (NOTICE, for example simplicity we will use the same image used for training and see how weights get activated")

print(f"Testing model for digit {image_label}:")
predictions = vector_matrix_multiplication(inputs, weights)
max_confidence = 0
digit_index= 0
for i in range(len(predictions)):
    if predictions[i] > max_confidence:
        max_confidence = predictions[i]
        digit_index = i

print(f"Model is {max_confidence*100:.1f}% confident that this is digit is {digit_index}")


--- Load MNIST dataset
Loading training images from: /home/rafal/work/repos/misc/book-grokking-deep-learning/resources/mnist/training-images-idx3-ubyte.gz
Dataset Shape: (60000, 28, 28)
Loading training labels from: /home/rafal/work/repos/misc/book-grokking-deep-learning/resources/mnist/training-labels-idx1-ubyte.gz
Dataset Shape: (60000,)

--- Print one of the images to confirm it's loaded correctly
Label for this digit: 5
image_bitmap shape: (28, 28)
                                                        
                                                        
                                                        
                                                        
                                                        
                        ░░░░░░░░▒▒▒▒▓▓░░▓▓████▒▒        
                ░░░░░░▓▓▓▓████████████▓▓████▓▓░░        
              ░░████████████████████░░░░░░░░░░          
              ░░████████████▓▓▓▓████                    
                ░░▓▓▒▒██████░

In [5]:
%load_ext autoreload
%autoreload 2

# Multiple Input and Multiple Outputs - Real World example: Classifier of handwritten digits
# Version 2.0 - train on multiple examples and classify any digit

# Input size: 784 (grayscale image representing handwritten digit with resolution of 28x28 pixels = 784 inputs)
# Output size: 10 (digits from 0 to 9)
# Training set size: 400

import numpy as np
np.set_printoptions(suppress=True) # supress scientific notation in printing
import resources.mnist.mnist_utils as mnist

# Math Utils
def weighted_sum(vector_a,vector_b):
    assert(len(vector_a) == len(vector_b))
    output = 0
    for i in range (len(vector_a)):
        output += vector_a[i].item() * vector_b[i].item()
    return output

def vector_matrix_multiplication(vector, matrix):
    assert(len(vector) == len(matrix[0]))
    output = np.zeros(len(matrix))
    for i in range(len(matrix)):
        output[i] = weighted_sum(vector, matrix[i])
    return output

def normalize_255_to_float(images):
    output = np.zeros((len(images), 784))
    for i in range(len(images)):
        arr_flat = images[i].flatten() # convert 28x28 matrix to 784 list
        arr_norm = arr_flat / 255.0 # normalize 0-255 value to 0-1 float
        output[i] = arr_norm
    return output

def build_training_outputs(training_labels):
    output = np.zeros((len(training_labels), 10))
    for i in range(len(training_labels)):
        output[i][training_labels[i]] = 1.0
    return output

def print_errors(errors):
    for i in range(len(errors)):
        print(f"Error for digit `{i}` = {errors[i]:.5f}")

def get_index_of_max(vector):
    max = 0
    index = 0
    for i in range(len(vector)):
        if vector[i] > max:
            max = vector[i]
            index = i
    return index

print("\n--- Load MNIST dataset")
training_images = mnist.load_training_images()[:200] # take only the first 100 images from a training set during debuging the code
training_labels = mnist.load_training_labels()[:200]


print("\n--- Print one of the images to confirm it's loaded correctly")
image_label = training_labels[0]
print(f"Label for this digit: {image_label}")
image_bitmap = training_images[0]
print(f"image_bitmap shape: {image_bitmap.shape}")
mnist.print_bitmap(image_bitmap)

normalized_training_images = normalize_255_to_float(training_images)

print(normalized_training_images.shape)

print("\n--- Print one of the normalized images to confirm it's loaded correctly")
mnist.print_float_image(normalized_training_images[0].reshape(28,28))

training_outputs = build_training_outputs(training_labels)

weights = np.zeros((10,784)) # experiment: let's see if how it learns with weights initialized with 0's

# Let the training algo run up to max 20 iterations
for iteration in range(10):
    errors = np.zeros(10)
    print(f"--- Iteration: {iteration}")
    for k in range(len(normalized_training_images)):
        inputs = normalized_training_images[k]
        outputs = training_outputs[k]
        deltas = np.zeros(10)
        weight_deltas = np.zeros((10,784))  # weight_deltas have the shape of weights matrix
        alpha = 0.01

        # Predict
        predictions = vector_matrix_multiplication(inputs, weights)

        # Figure out the Error and update Weights
        for i in range(len(outputs)):
            errors[i] = (predictions[i] - outputs[i]) ** 2
            deltas[i] = predictions[i] - outputs[i]
            for j in range(len(inputs)):
                weight_deltas[i][j] = deltas[i].item() * inputs[j].item()
                weights[i][j] -= alpha * weight_deltas[i][j]

    print_errors(errors)
     # Evaluate Error
    if np.all(errors < 0.01):
        print(f"=== Training finished after {iteration} iterations All errors are below 0.1% !")
        break

print("\n=== Lets peek into weights for detecting our digits - we should be able to see some kind of composite image of all possible ways to draw each digit (from our training set)")
for i in range(len(weights)):
    print(f"\n\n---\nWeights for detecting Digit `{i}`")
    mnist.print_weights_image(weights[i].reshape(28,28))

print("\n==== Test time! Lets see how many of 100 images our model will clasify correclty")

test_images = mnist.load_test_images()[:100]
test_labels = mnist.load_test_labels()[:100]

normalized_test_images = normalize_255_to_float(test_images)

success_count = 0
for i in range(len(test_images)):
    test_image = test_images[i]
    inputs = normalized_test_images[i]

    predictions = vector_matrix_multiplication(inputs, weights)
    if test_labels[i] == get_index_of_max(predictions):
        success_count += 1

print(F"FINAL RESULTS: Our model classified correctly {success_count} of {len(test_images)} test images!")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Load MNIST dataset
Loading training images from: /home/rafal/work/repos/misc/book-grokking-deep-learning/resources/mnist/training-images-idx3-ubyte.gz
Dataset Shape: (60000, 28, 28)
Loading training labels from: /home/rafal/work/repos/misc/book-grokking-deep-learning/resources/mnist/training-labels-idx1-ubyte.gz
Dataset Shape: (60000,)

--- Print one of the images to confirm it's loaded correctly
Label for this digit: 5
image_bitmap shape: (28, 28)
                                                        
                                                        
                                                        
                                                        
                                                        
                        ░░░░░░░░▒▒▒▒▓▓░░▓▓████▒▒        
                ░░░░░░▓▓▓▓████████████▓▓████▓▓░░        
              ░░████████████████████░░░░░░░░░░         